<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_2_ROC_AUC_Threshold_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROC Curves, AUC, and Threshold Tuning

**Author:** Brad Sheese

---

## Introduction

Building on Notebook 2's confusion matrix and metrics, this notebook explores:

- **ROC Curve**: Visualizes the trade-off between TPR and FPR at ALL thresholds
- **AUC**: Area Under the ROC Curve - a threshold-independent measure
- **Youden's J**: Finding the optimal operating point
- **Precision-Recall Curve**: Better for imbalanced data

### Learning Objectives
By the end of this notebook, you will:
1. Interpret ROC curves and understand the FPR/Specificity relationship
2. Calculate and interpret AUC as an integration
3. Use Youden's J to find the optimal threshold
4. Know when to use Precision-Recall curves instead of ROC

## Setup and Train Model

Recall from Notebook 2: The confusion matrix depends on threshold. Let's see how metrics change across all possible thresholds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load Credit Default dataset
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Test set: {len(y_test)} samples, {y_test.mean()*100:.1f}% defaults")

## ROC Curve: TPR vs FPR at Every Threshold

The **ROC (Receiver Operating Characteristic) Curve** plots:

| Axis | Metric | Formula | Meaning |
|------|--------|---------|---------|
| **Y-axis** | True Positive Rate (TPR) | TP/(TP+FN) | Recall - how many positives did we catch? |
| **X-axis** | False Positive Rate (FPR) | FP/(TN+FP) | 1 - Specificity - how many negatives wrongly flagged? |

**Key relationship:** FPR = 1 - Specificity

As we lower the threshold:
- We catch more positives (higher TPR) BUT
- We also wrongly flag more negatives (higher FPR)

The curve traces this trade-off at every possible threshold (0 to 1).

In [ ]:
# Compute ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

print(f"AUC Score: {roc_auc:.3f}")
print(f"Interpretation: {roc_auc*100:.1f}% chance the model ranks a random positive above a random negative")
print(f"\nAUC Formula: ∫ ROC curve dFPR (area under the curve)")
print("  - AUC = 0.5 → random classifier (diagonal)")
print("  - AUC = 1.0 → perfect classifier (hugs top-left)")
print("  - AUC > 0.8 → generally good discriminative ability")

# Plot ROC curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Recall/Sensitivity)')
ax.set_title('ROC Curve: Credit Default Prediction')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Youden's J: Finding Optimal Threshold

Youden's J statistic finds the threshold that best balances TPR and FPR:

$$J = Maximize(TPR - FPR) = Maximize(TPR + TNR - 1) = Maximize(TPR + Specificity - 1)$$

- J = 1 = perfect (both TPR=1 and FPR=0)
- J = 0 = no better than random
- The optimal threshold maximizes TPR - FPR

This corresponds to the point on the ROC curve farthest from the diagonal.

In [ ]:
# Calculate Youden's J for each point
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds_roc[optimal_idx]
optimal_tpr = tpr[optimal_idx]
optimal_fpr = fpr[optimal_idx]

print(f"Youden's J optimal point:")
print(f"  Threshold: {optimal_threshold:.3f}")
print(f"  TPR (Recall): {optimal_tpr:.3f}")
print(f"  FPR: {optimal_fpr:.3f}")
print(f"  J = {j_scores[optimal_idx]:.3f}")

# Plot with optimal point
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.5)
ax.scatter([optimal_fpr], [optimal_tpr], color='red', s=100, zorder=5, label=f'Optimal (J={j_scores[optimal_idx]:.3f})')
ax.axhline(y=optimal_tpr, color='red', linestyle=':', alpha=0.3)
ax.axvline(x=optimal_fpr, color='red', linestyle=':', alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title("ROC Curve with Youden's J Optimal Point")
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Precision-Recall Curve: Better for Imbalanced Data

**Why not just use ROC for imbalanced data?**

ROC's FPR can be misleading when negatives vastly outnumber positives:
- Even a model with many false positives might have low FPR if TN is huge
- ROC can look great (high AUC) even when recall is terrible

**Precision-Recall Curve** shows directly:
- **Y-axis**: Precision = TP/(TP+FP)
- **X-axis**: Recall = TP/(TP+FN)
- More honest about performance on imbalanced data
- Baseline: horizontal line at % positives (random baseline)

**PR-AUC interpretation:**
- Higher is better
- Compare to baseline (always predicting majority class)

In [ ]:
# Compute Precision-Recall curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)

# Calculate PR-AUC correctly (precision as function of recall)
sorted_indices = np.argsort(recall)
recall_sorted = recall[sorted_indices]
precision_sorted = precision[sorted_indices]
pr_auc = auc(recall_sorted, precision_sorted)

baseline = y_test.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='blue', lw=2, label=f'PR Curve (AUC = {pr_auc:.3f})')
ax.axhline(y=baseline, color='red', linestyle='--', label=f'Baseline (always negative): {baseline:.2f}')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve: Credit Default Prediction')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PR-AUC: {pr_auc:.3f}")
print(f"Baseline (no skill): {baseline:.3f}")
print(f"\nOur model achieves {pr_auc - baseline:.3f} above random baseline")
print(f"(For imbalanced data, PR-AUC is more informative than ROC-AUC)")

## Threshold Tuning: Finding the Optimal Decision Boundary

The default threshold (0.5) may not be optimal. Let's explore different thresholds and their impact on metrics.

Recall from Notebook 2:
- Lower threshold → more positives → Higher recall, Lower precision
- Higher threshold → fewer positives → Lower recall, Higher precision

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Test different thresholds
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []
for thresh in thresholds_to_test:
    y_pred_t = (y_proba >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    n_positives = y_pred_t.sum()
    results.append({
        'threshold': thresh, 
        'predicted_positives': n_positives,
        'precision': prec, 
        'recall': rec, 
        'f1': f1
    })

results_df = pd.DataFrame(results)
print("Performance at Different Thresholds:")
print(results_df.to_string(index=False))

# Find best F1
best_idx = results_df['f1'].idxmax()
best_thresh = results_df.loc[best_idx, 'threshold']
print(f"\nBest F1-Score ({results_df.loc[best_idx, 'f1']:.3f}) at threshold = {best_thresh}")
print(f"Youden's J optimal threshold (from ROC): {optimal_threshold:.3f}")

In [ ]:
# Visualize metrics vs threshold
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df['threshold'], results_df['precision'], 'o-', label='Precision', linewidth=2, color='green')
ax.plot(results_df['threshold'], results_df['recall'], 's-', label='Recall', linewidth=2, color='blue')
ax.plot(results_df['threshold'], results_df['f1'], '^-', label='F1-Score', linewidth=2, color='red')
ax.axvline(x=best_thresh, color='red', linestyle=':', alpha=0.7, label=f'Best F1 threshold={best_thresh}')
ax.axvline(x=optimal_threshold, color='orange', linestyle=':', alpha=0.7, label=f'Youden J threshold={optimal_threshold:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision, Recall, F1 vs. Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Business Context: Choosing Threshold Based on Costs

In practice, the "best" threshold depends on your business costs:
- **False Positive (FP)**: Approving a risky loan (costs money)
- **False Negative (FN)**: Rejecting a good customer (missed opportunity)

Let's assume missing a default costs **10x more** than a false alarm.

In [ ]:
# Calculate business cost at each threshold
# Cost = FP + (10 * FN) normalized
cost_ratio = 10  # Cost of FN is 10x cost of FP

costs = []
for thresh in thresholds_to_test:
    y_pred_t = (y_proba >= thresh).astype(int)
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    cost = (fp + cost_ratio * fn) / len(y_test)
    costs.append({'threshold': thresh, 'cost': cost})

costs_df = pd.DataFrame(costs)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(costs_df['threshold'], costs_df['cost'], 'o-', color='crimson', linewidth=2, markersize=8)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Normalized Cost')
ax.set_title(f'Business Cost at Different Thresholds (FN costs {cost_ratio}x FP)')
ax.grid(True, alpha=0.3)

optimal_thresh = costs_df.loc[costs_df['cost'].idxmin(), 'threshold']
ax.axvline(x=optimal_thresh, color='green', linestyle='--', label=f'Optimal: {optimal_thresh}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal threshold (given cost ratio {cost_ratio}): {optimal_thresh}")
print("\nKey insight: Adjust threshold based on YOUR business costs, not just F1!")

## Key Takeaways

1. **ROC-AUC** summarizes overall discrimination ability across all thresholds
2. **Youden's J** finds optimal threshold: maximizes (TPR - FPR) = maximizes (TPR + Specificity - 1)
3. **PR-AUC** is more informative for imbalanced data - always plot the PR curve!
4. **Threshold matters**: The optimal threshold depends on your business context
5. **Cost-sensitive tuning**: Choose threshold to minimize expected cost, not just maximize F1
6. **Trade-offs**: Lower threshold = more recall, fewer precision (and vice versa)

In Notebook 4, we'll explore model selection and introduce multiclass classification.